<a href="https://colab.research.google.com/github/RohanKJoseph/IPRS_Tasks/blob/main/NumPy_TASK_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Gaming Performance Analytics


In [1]:
import numpy as np

In [2]:
# Load dataset
data = np.array([
    [501, 45, 3, 120, 60],
    [502, 30, 2, 80, 50],
    [503, 60, 4, np.nan, 70],
    [504, 25, 1, 60, 40],
    [505, 80, 5, 200, 90],
    [506, -20, 2, 70, 50],
    [507, 35, 2, 90, 55],
    [508, 50, 3, np.nan, 65],
    [509, 90, 6, 250, 100],
    [510, 200, 4, 150, 80],
    [511, 55, 3, 110, 60],
    [512, 40, 2, 95, 55]
])

## Understanding Dataset

In [3]:
rows, cols = data.shape
print(f"Dataset Shape: {data.shape}")
print(f"Number of Rows (Players): {rows}")
print(f"Number of Columns (Features): {cols}")

Dataset Shape: (12, 5)
Number of Rows (Players): 12
Number of Columns (Features): 5


##Insight Task:

###A good player exhibits high efficiency rather than just high session duration. Key metrics include:Completion Rate / Speed: Completing more levels in less time ($\frac{\text{levels\_completed}}{\text{session\_duration}}$).Action Efficiency: Achieving progress with optimized actions rather than idle clicks.Resource Management: Maximizing level progression per unit of energy_used.

###Key Factors Affecting Performance are
Session Length: Extremely long sessions (>150 mins) cause fatigue; short sessions (<20 mins) lack progress.

Action Rate (Pacing): Higher actions per minute directly correlate with faster level completions.

Data Integrity: Anomalies (negative duration, AFK idling, missing telemetry) heavily distort performance analytics.

##Indexing & Slicing

In [6]:
#Extract specific columns
player_ids = data[:, 0]
print("Players IDs",player_ids)
session_durations = data[:, 1]
print("Section durations: ",session_durations)


Players IDs [501. 502. 503. 504. 505. 506. 507. 508. 509. 510. 511. 512.]
Section durations:  [ 45.  30.  60.  25.  80. -20.  35.  50.  90. 200.  55.  40.]


In [8]:
#Slice first 5 players
first_5_players = data[:5, :]
print("First 5 players: \n",first_5_players)

First 5 players: 
 [[501.  45.   3. 120.  60.]
 [502.  30.   2.  80.  50.]
 [503.  60.   4.  nan  70.]
 [504.  25.   1.  60.  40.]
 [505.  80.   5. 200.  90.]]


In [10]:
# Access one player
print("First player data: ",data[0, :])

First player data:  [501.  45.   3. 120.  60.]


##Basic Analysis

In [15]:
# Basic statistics using NaN-aware functions
print(f"Average Session Duration: {np.nanmean(data[:, 1]):.2f} minutes")
print(f"Max Session Duration: {np.nanmax(data[:, 1])} minutes")
print(f"Min Session Duration: {np.nanmin(data[:, 1])} minutes")
print(f"Average Levels Completed: {np.nanmean(data[:, 2]):.2f}")
print(f"Total In-Game Actions: {np.nansum(data[:, 3]):.0f}")

Average Session Duration: 57.50 minutes
Max Session Duration: 200.0 minutes
Min Session Duration: -20.0 minutes
Average Levels Completed: 3.08
Total In-Game Actions: 1225


#Insight Task

###1. Unrealistic Playtime Anomaly: Player 506 shows a duration of -20 minutes, indicating logging errors or client telemetry failure.

###2. Afk / Outlier Player: Player 510 played for 200 minutes (over 3 hours), which is more than 3$\times$ the average playtime, pointing to potential idling (AFK).

###3. Strong Action Correlation: Players 505 and 509 logged the highest actions (200 & 250) and completed the highest levels (5 & 6), confirming that active play directly yields progress.

###4. Missing Telemetry Data: Players 503 and 508 have missing (NaN) action data despite completing 4 and 3 levels, highlighting missing metric collection.

###5 .Short Sessions yield Low Progress: Player 504 spent only 25 minutes and completed only 1 level, reflecting low engagement or early user churn.

##Vectorization

In [19]:
# Add bonus actions (+10)
actions_with_bonus = data[:, 3] + 10
print("Bonus added: ",actions_with_bonus)

Bonus added:  [130.  90.  nan  70. 210.  80. 100.  nan 260. 160. 120. 105.]


In [27]:
#Identify high performers (> 4 levels)
high_performer_mask = data[:, 2] > 4
high_performers = data[high_performer_mask]
print("Data of High performers are: \n",high_performers)

Data of High performers are: 
 [[505.  80.   5. 200.  90.]
 [509.  90.   6. 250. 100.]]


In [26]:
#Identify low activity players (<80 actions)
low_activity_mask = data[:, 3] < 80
low_activity_players = data[low_activity_mask]
print("Data of Low activity players are: \n",low_activity_players)

Data of Low activity players are: 
 [[504.  25.   1.  60.  40.]
 [506. -20.   2.  70.  50.]]


#Insight Task

###Top Performers Concentration: Only 2 out of 12 players (505 and 509) qualified as high performers (>4 levels completed), representing ~16.7% of the player base.

###Low Activity Threshold: Players 504 and 506 logged under 80 actions, identifying them as candidates for re-engagement or tutorial onboarding.

###Vectorization Efficiency: Operations executed instantly without Python for loops, leveraging SIMD operations in C under NumPy.

## Data Cleaning


In [29]:
cleaned_data = data.copy() # copyin the entire data to a new variable

# Handle negative values
cleaned_data[cleaned_data[:, 1] <= 0, 1] = np.nan
#Detect and handle outliers using mean + 2 * std
dur_mean = np.nanmean(cleaned_data[:, 1])
dur_std = np.nanstd(cleaned_data[:, 1])
upper_bound = dur_mean + 2 * dur_std


outlier_mask = cleaned_data[:, 1] > upper_bound
cleaned_data[outlier_mask, 1] = np.nan
print(f"Session Duration Upper Bound (Mean + 2*STD): {upper_bound:.2f} mins")

# Replace missing values with mean
col_means = np.nanmean(cleaned_data, axis=0)

for col_idx in range(1, cleaned_data.shape[1]):
    nan_mask = np.isnan(cleaned_data[:, col_idx])
    cleaned_data[nan_mask, col_idx] = col_means[col_idx]

      #data cleaned properly

Session Duration Upper Bound (Mean + 2*STD): 158.30 mins


In [34]:
#• Create Performance Score = levels_completed / session_duration

performance_score = cleaned_data[:, 2] / cleaned_data[:, 1]

#• Add using column_stack

final_dataset = np.column_stack((cleaned_data, performance_score))

print("\n--- Cleaned & Feature-Engineered Dataset ---")
np.set_printoptions(suppress=True, precision=4)
print(final_dataset)


--- Cleaned & Feature-Engineered Dataset ---
[[501.      45.       3.     120.      60.       0.0667]
 [502.      30.       2.      80.      50.       0.0667]
 [503.      60.       4.     122.5     70.       0.0667]
 [504.      25.       1.      60.      40.       0.04  ]
 [505.      80.       5.     200.      90.       0.0625]
 [506.      51.       2.      70.      50.       0.0392]
 [507.      35.       2.      90.      55.       0.0571]
 [508.      50.       3.     122.5     65.       0.06  ]
 [509.      90.       6.     250.     100.       0.0667]
 [510.      51.       4.     150.      80.       0.0784]
 [511.      55.       3.     110.      60.       0.0545]
 [512.      40.       2.      95.      55.       0.05  ]]


##Final Insights


In [35]:
#  Energy Efficiency Ratio
energy_efficiency = final_dataset[:, 2] / final_dataset[:, 4]

# Ranking players based on Performance Score
sorted_indices = np.argsort(final_dataset[:, 5])[::-1] # Descending order
ranked_players = final_dataset[sorted_indices]

# Overall Statistics Summary
print("\n--- Top 3 Best Performing Players ---")
for player in ranked_players[:3]:
    print(f"Player ID: {int(player[0])} | Score: {player[5]:.4f} | Levels: {int(player[2])} | Duration: {player[1]:.1f}m")

print(f"\nOverall Average Performance Score: {np.mean(performance_score):.4f}")
print(f"Overall Average Energy Efficiency: {np.mean(energy_efficiency):.4f} levels/energy")


--- Top 3 Best Performing Players ---
Player ID: 510 | Score: 0.0784 | Levels: 4 | Duration: 51.0m
Player ID: 509 | Score: 0.0667 | Levels: 6 | Duration: 90.0m
Player ID: 501 | Score: 0.0667 | Levels: 3 | Duration: 45.0m

Overall Average Performance Score: 0.0590
Overall Average Energy Efficiency: 0.0455 levels/energy
